# Decision Modeling Project


## Step 1:

### Formulation of the problem:

#### Variables, Sets, Data:
- $x_{i,j} \in \{0,1\}$ if brick $i$ is assigned to SR $j$
- $d_{i,j}$ = distance between brick $i$ and SR $j$ (data)
- $v_i$ = workload for brick $i$ (value between 0 and 1) (data)
- $S$ = set of SRs
- $B$ = set of bricks $(1, 2, 3, \ldots)$

#### Constraints (30 in total):

$\sum_{i \in B} x_{i,1} \cdot v_i \leq 1.2$

$\sum_{i \in B} x_{i,2} \cdot v_i \leq 1.2$

$\sum_{i \in B} x_{i,3} \cdot v_i \leq 1.2$

$\sum_{i \in B} x_{i,4} \cdot v_i \leq 1.2$

$\sum_{i \in B} x_{i,1} \cdot v_i \geq 0.8$

$\sum_{i \in B} x_{i,2} \cdot v_i \geq 0.8$

$\sum_{i \in B} x_{i,3} \cdot v_i \geq 0.8$

$\sum_{i \in B} x_{i,4} \cdot v_i \geq 0.8$

22 times this constraints for each brick i:

$\sum_{j \in S} x_{1,j} = 1$ (example with i=1)

#### Objective Function:

$$\min \sum_{i \in B} \sum_{j \in S} d_{i,j} \cdot x_{i,j}$$

$$\min \sum_{i \in B} \sum_{j \in S} \left({x'}_{i,j} - 2{x'}_{i,j} \cdot x_{i,j} + x_{i,j}\right)$$

### Using GUROBI for 22 Bricks and 4 SR

In [6]:
# !pip install gurobipy

In [ ]:
import gurobipy
from gurobipy import Model, GRB
import matplotlib.pyplot as plt

#  implement your two mono-objective models using GUROBI, and solve the instance with 22 Bricks
# and 4 Sales Representatives

model = Model("Bricks_and_Sales_Reps")
num_bricks = 22
num_SR = 4

# --- Data ---
distances = [[16.16, 24.08, 24.32, 21.12], 
     [19, 26.47, 27.24, 17.33], 
     [25.29, 32.49, 33.42, 12.25], 
     [0, 7.93, 8.31, 36.12], 
     [3.07, 6.44, 7.56, 37.37], 
     [1.22, 7.51, 8.19, 36.29], 
     [2.80, 10.31, 10.95, 33.5], 
     [2.87, 5.07, 5.67, 38.8], 
     [3.8, 8.01, 7.41, 38.16], 
     [12.35, 4.52, 4.35, 48.27], 
     [11.11, 3.48, 2.97, 47.14], 
     [21.99, 22.02, 24.07, 39.86], 
     [8.82, 3.3, 5.36, 43.31], 
     [7.93, 0, 2.07, 43.75], 
     [9.34, 2.25, 1.11, 45.43], 
     [8.31, 2.07, 0, 44.43], 
     [7.31, 2.44, 1.11, 43.43], 
     [7.55, 0.75, 1.53, 43.52], 
     [11.13, 18.41, 19.26, 25.4], 
     [17.49, 23.44, 24.76, 23.21], 
     [11.03, 18.93, 19.28, 25.43], 
     [36.12, 43.75, 44.43, 0]] 

workloads = [0.1609, 0.1164, 0.1026, 0.1516, 0.0939, 
     0.1320, 0.0687, 0.0930, 0.2116, 0.2529, 
     0.0868, 0.0828, 0.0975, 0.8177, 0.4115, 
     0.3795, 0.0710, 0.0427, 0.1043, 0.0997, 
     0.1698, 0.2531]



# --- Variables ---
x = model.addVars(num_bricks, num_SR, vtype=GRB.BINARY, name="x")

# --- Constraints ---
for j in range(num_SR):
    model.addConstr(sum(x[i,j] * workloads[i] for i in range(num_bricks)) <= 1.2, name=f"max_workload_SR{j+1}")
    model.addConstr(sum(x[i,j] * workloads[i] for i in range(num_bricks)) >= 0.8, name=f"min_workload_SR{j+1}")

for i in range(num_bricks):
    model.addConstr(sum(x[i,j] for j in range(num_SR)) == 1, name=f"brick_{i+1}_assignment")

# --- Objective function 1 : Minimizing the distance ---
model.setObjective(sum(distances[i][j] * x[i,j] for i in range(num_bricks) for j in range(num_SR)),GRB.MINIMIZE)

# --- Mapping précédent (0-based) ---
prev = [None]*22
for b in [4,5,6,7,8,15]:              prev[b-1] = 0
for b in [10,11,12,13,14]:            prev[b-1] = 1
for b in [9,16,17,18]:                prev[b-1] = 2
for b in [1,2,3,19,20,21,22]:         prev[b-1] = 3
assert all(p is not None for p in prev)

# --- Matrice one-hot x_prev ---
x_prev = [[1 if j == prev[i] else 0 for j in range(num_SR)] for i in range(num_bricks)]



# --- Resolve the optimization problem ---
model.Params.OutputFlag = 0
model.optimize()

In [8]:
for j in range(num_SR):
    assigned = [i+1 for i in range(num_bricks) if x[i,j].X > 0.5]
    print(f"SR {j+1}: {assigned}")

SR 1: [4, 5, 6, 7, 8, 9, 12, 19, 20]
SR 2: [11, 13, 14, 18]
SR 3: [10, 15, 16, 17]
SR 4: [1, 2, 3, 21, 22]


In [9]:
# # --- Objective function 2 : Minimize the disruption ---
# # x_prev[i][j] = 0/1 historique (one-hot sur j)
# model.setObjective(
#     sum(
#         (x[i,j])                       # x_after^2 -> x_after
#         - 2 * x_prev[i][j] * x[i,j]    # -2 x_after x_before
#         + x_prev[i][j]                 # x_before^2 -> x_before (constant)
#         for i in range(num_bricks) for j in range(num_SR)
#     ),
#     GRB.MINIMIZE
# )

In [10]:
# print("\n=== Solution Disruption ===")
# changes = sum(1 - int(x[i, prev[i]].X + 1e-9) for i in range(num_bricks))
# print(f"Disruption (nb de bricks réaffectés) = {changes}")
# for i in range(num_bricks):
#     old_j = prev[i]
#     new_j = next(j for j in range(num_SR) if x[i,j].X > 0.5)
#     if new_j != old_j:
#         print(f"Brick {i+1}: SR{old_j+1} -> SR{new_j+1}")



In [ ]:
# import gurobipy as gp
# from gurobipy import GRB

# def build_model(distances, workloads, L=0.8, U=1.2, prev=None):
#     nB = len(workloads)
#     nS = len(distances[0])
#     m = gp.Model()
#     x = m.addVars(nB, nS, vtype=GRB.BINARY, name="x")

#     # Affectation unique
#     for i in range(nB):
#         m.addConstr(gp.quicksum(x[i,j] for j in range(nS)) == 1)
#     # Charges
#     for j in range(nS):
#         m.addConstr(gp.quicksum(workloads[i]*x[i,j] for i in range(nB)) >= L)
#         m.addConstr(gp.quicksum(workloads[i]*x[i,j] for i in range(nB)) <= U)

#     # Expressions utiles
#     dist_expr = gp.quicksum(distances[i][j]*x[i,j] for i in range(nB) for j in range(nS))
#     if prev is not None:
#         disr_expr = gp.quicksum(1 - x[i, prev[i]] for i in range(nB))
#     else:
#         disr_expr = None
#     return m, x, dist_expr, disr_expr

# def solve_disruption_bound(distances, workloads, prev, L=0.8, U=1.2, maximize=False):
#     m, x, _, disr = build_model(distances, workloads, L, U, prev)
#     m.Params.OutputFlag = 0
#     if maximize:
#         m.setObjective(disr, GRB.MAXIMIZE)
#     else:
#         m.setObjective(disr, GRB.MINIMIZE)
#     m.optimize()
#     return int(round(m.objVal))  # disruption est entière

# def epsilon_constraint_front(distances, workloads, prev, L=0.8, U=1.2, lam=1e-6):
#     dmin = solve_disruption_bound(distances, workloads, prev, L, U, maximize=False)
#     dmax = solve_disruption_bound(distances, workloads, prev, L, U, maximize=True)

#     pareto = []           # (disruption, distance, assignment_tuple)
#     seen_assignments = set()

#     for eps in range(dmin, dmax+1):
#         m, x, dist, disr = build_model(distances, workloads, L, U, prev)
#         m.Params.OutputFlag = 0
#         # contrainte epsilon
#         m.addConstr(disr <= eps)
#         # objectif avec tie-break pour efficacité
#         m.setObjective(dist + lam*disr, GRB.MINIMIZE)
#         m.optimize()
#         if m.status == GRB.OPTIMAL:
#             assign = tuple(int(round(x[i,j].X)) for i in range(len(workloads)) for j in range(len(distances[0])))
#             if assign in seen_assignments:
#                 continue
#             seen_assignments.add(assign)
#             pareto.append((int(round(disr.getValue())), dist.getValue(), assign))

#     # filtre Pareto au cas où (sécurité)
#     pareto_nd = []
#     for d1, f1, a1 in pareto:
#         if not any((d2 <= d1 and f2 <= f1) and (d2 < d1 or f2 < f1)
#                    for d2, f2, _ in pareto):
#             pareto_nd.append((d1, f1, a1))
#     # tri par disruption puis distance
#     pareto_nd.sort(key=lambda t: (t[0], t[1]))
#     return pareto_nd
